In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.7 MB/s eta 0:00:00a 0:00:01


In [74]:
import shutil, os

shutil.rmtree("dataset/labels")

os.makedirs("dataset/labels/train", exist_ok=True)
os.makedirs("dataset/labels/val", exist_ok=True)
os.makedirs("dataset/labels/test", exist_ok=True)

In [75]:
img_size = 1024

all_images = set(
    [f.replace(".jpg","") for f in os.listdir("dataset/images/train")] +
    [f.replace(".jpg","") for f in os.listdir("dataset/images/val")] +
    [f.replace(".jpg","") for f in os.listdir("dataset/images/test")]
)

for _, row in df.iterrows():

    image_id = row["image_id"]

    if image_id not in all_images:
        continue

    class_name = row["class_name"]

    if class_name not in class_map:
        continue

    x_min = row["x_min"]
    y_min = row["y_min"]
    x_max = row["x_max"]
    y_max = row["y_max"]

    class_id = class_map[class_name]

    x_center = ((x_min + x_max) / 2) / img_size
    y_center = ((y_min + y_max) / 2) / img_size
    width = (x_max - x_min) / img_size
    height = (y_max - y_min) / img_size

    # clip values
    x_center = max(0, min(x_center, 0.999))
    y_center = max(0, min(y_center, 0.999))
    width = max(0, min(width, 0.999))
    height = max(0, min(height, 0.999))

    label_line = f"{class_id} {x_center} {y_center} {width} {height}\n"

    if image_id in train_ids:
        folder = "train"
    elif image_id in val_ids:
        folder = "val"
    else:
        folder = "test"

    label_path = f"dataset/labels/{folder}/{image_id}.txt"

    with open(label_path, "a") as f:
        f.write(label_line)

In [76]:
for img in os.listdir("dataset/images/train"):

    label_name = img.replace(".jpg",".txt")
    path = f"dataset/labels/train/{label_name}"

    if not os.path.exists(path):
        open(path,"w").close()

In [77]:
def copy_labels(image_list, folder):

    for img in image_list:

        label_name = img.replace(".jpg",".txt")

        src = f"dataset/labels/train/{label_name}"
        dst = f"dataset/labels/{folder}/{label_name}"

        if os.path.exists(src):
            shutil.copy(src,dst)
        else:
            open(dst,"w").close()

copy_labels(val,"val")
copy_labels(test,"test")

In [66]:
count = 0

for file in os.listdir("dataset/labels/train"):
    with open(f"dataset/labels/train/{file}") as f:
        content = f.read().strip()
        if content != "":
            print(file, ":", content)
            count += 1
            if count == 5:
                break

54d9b89fd235a6141bc3512482e988a9.txt : 2 0.80517578125 1 0.2333984375 0.1640625
2 0.80322265625 1 0.2451171875 0.181640625
2 0.79248046875 1 0.1435546875 0.130859375
19da7247b897983617e17c69636620ad.txt : 2 0.71240234375 0.9365234375 0.3466796875 0.24609375
2 0.63134765625 0.91015625 0.6103515625 0.5546875
2 1 1 0.427734375 0.7734375
3 1 1 0.427734375 0.7734375
2 0.6328125 0.8779296875 0.705078125 0.591796875
2 1 0.67138671875 0.1669921875 0.2373046875
4e7a534c72ee26ef1610dd6e5c94d246.txt : 2 1 1 0.54296875 0.3408203125
2 1 1 0.5087890625 0.310546875
2f3bb25eb6ae1a19b982c6ba56f725e2.txt : 2 0.5537109375 0.96826171875 0.591796875 1
2 0.50341796875 1 0.7802734375 1
2 1 0.79052734375 0.5986328125 0.7080078125
2 1 1 0.71484375 1
26585204e3c296a3b105bd5bd1c537ee.txt : 2 1 1 0.09765625 0.048828125
1 1 1 0.1591796875 0.185546875


In [80]:
import os
import shutil

train_label_dir = "dataset/labels/train"
val_label_dir = "dataset/labels/val"
test_label_dir = "dataset/labels/test"

train_img_dir = "dataset/images/train"
val_img_dir = "dataset/images/val"
test_img_dir = "dataset/images/test"

count = 0

for label in os.listdir(train_label_dir):
    
    path = os.path.join(train_label_dir, label)

    if os.path.getsize(path) > 0:   # image with abnormality
        
        img = label.replace(".txt", ".jpg")
        
        shutil.move(os.path.join(train_img_dir, img),
                    os.path.join(val_img_dir, img))
        
        shutil.move(path,
                    os.path.join(val_label_dir, label))
        
        count += 1
        
        if count == 100:   # move 100 abnormal images
            break

print("Moved abnormal images to validation:", count)

Moved abnormal images to validation: 100


In [81]:
count = 0
for file in os.listdir("dataset/labels/val"):
    if os.path.getsize(f"dataset/labels/val/{file}") > 0:
        count += 1

print("Validation images with boxes:", count)

Validation images with boxes: 200


In [82]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="config.yaml",
    epochs=50,
    imgsz=640,
    batch=8
)

New https://pypi.org/project/ultralytics/8.4.20 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d7c77ce6390>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04